# Notebook 1B — Inference via Kaggle GPU (vLLM)
**MedThink-Bench VI Eval Pipeline**

Output: `checkpoint_inference.jsonl` → cùng unified schema với Notebook 1A

---
**Yêu cầu**: Kaggle accelerator = GPU T4 x2 
`pip install vllm`

In [ ]:
# ── Cell 1: Install dependencies ────────────────────────────────────────
!pip install vllm -q

In [ ]:
import torch
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {free/1e9:.1f}/{total/1e9:.1f} GB free")

In [ ]:
# ── Cell 1.5: Copy shared modules từ dataset vào working dir ────────────
import shutil, os

_SRC = "/kaggle/input/datasets/quangminh2401/medthink-vi-eval-pipeline"
_DST = "/kaggle/working"

for _f in ["utils.py", "async_openrouter.py", "evaluator.py"]:
    shutil.copy(os.path.join(_SRC, _f), os.path.join(_DST, _f))
    print(f"copied: {_f}")

In [ ]:
# ── Cell 2: CONFIG - Điền thông tin cấu hình tại đây ───────────────────────────────────────────────────────
import sys
sys.path.insert(0, "/kaggle/working")


# ── Đường dẫn file ──────────────────────────
DATASET_PATH        = "/kaggle/input/datasets/quangminh2401/medthink-benchfull-translated/vi_QA_data.jsonl"   # Dataset tiếng Việt
CHECKPOINT_INFER    = "/kaggle/working/checkpoint_inference.jsonl"    # Resume inference
CHECKPOINT_EVAL     = "/kaggle/working/checkpoint_eval.jsonl"         # Resume eval
OUTPUT_EVAL         = "/kaggle/working/results_eval.jsonl"            # Kết quả per-sample
OUTPUT_SUMMARY      = "/kaggle/working/summary_metrics.json"          # Aggregate report

# ── OpenRouter API ───────────────────────────
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1/chat/completions"


# ── Model inference (Notebook 1B – Kaggle GPU/TPU) ──
INFER_TPU_MODEL_ID  = "FreedomIntelligence/HuatuoGPT-o1-8B"                     # HuggingFace model ID
INFER_TPU_TENSOR_PARALLEL = 2                                          # Số GPU (T4 x2 = 2)
INFER_TPU_MAX_TOKENS      = 4096
INFER_TPU_TEMPERATURE     = 0.0
INFER_TPU_BATCH_SIZE      = 16

# ── Prompt inference ─────────────────────────
INFER_SYSTEM_PROMPT = """
Bạn là một bác sĩ chuyên khoa đang trả lời câu hỏi trắc nghiệm y khoa.
Hãy suy luận từng bước, sau đó đưa ra đáp án cuối cùng.

Trả lời ĐÚNG theo định dạng JSON sau, không thêm bất kỳ nội dung nào khác:
{"answer": "<chỉ một chữ cái>", "reasoning": "<lý luận từng bước của bạn>"}"""

INFER_USER_TEMPLATE = """{question}"""

print(f"Model   : {INFER_TPU_MODEL_ID}")
print(f"Dataset : {DATASET_PATH}")
print(f"Output  : {CHECKPOINT_INFER}")
print(f"Tensor parallel: {INFER_TPU_TENSOR_PARALLEL}")

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

# Set env var TRƯỚC khi import vllm hoặc load model
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN  # legacy key, một số lib vẫn dùng

import huggingface_hub
huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)

# Verify
info = huggingface_hub.model_info("FreedomIntelligence/HuatuoGPT-o1-8B")
print(f"Access OK: {info.id}")

In [ ]:
# ── Cell 3: Load model ──────────────────────────────────────────────────
import os
import subprocess

# Fix libcuda symlink cho FlashInfer linker
os.system("ln -sf /usr/local/cuda-12.8/compat/libcuda.so /usr/lib/x86_64-linux-gnu/libcuda.so")
os.system("ln -sf /usr/local/cuda-12.8/compat/libcuda.so /usr/lib/x86_64-linux-gnu/libcuda.so.1")
os.system("ldconfig")

# Set env var TRƯỚC khi import vllm
os.environ["VLLM_ATTENTION_BACKEND"]       = "XFORMERS"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"]  = "0"

import torch
from vllm import LLM, SamplingParams

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {free/1e9:.1f}/{total/1e9:.1f} GB free")

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

llm = LLM(
    model                    = INFER_TPU_MODEL_ID,
    tensor_parallel_size     = INFER_TPU_TENSOR_PARALLEL,
    dtype                    = "float16",
    trust_remote_code        = False,   
    max_model_len            = 3072,    
    gpu_memory_utilization   = 0.8,    
    disable_custom_all_reduce= True,
    enforce_eager            = True,
)

sampling_params = SamplingParams(
    temperature = INFER_TPU_TEMPERATURE,
    max_tokens  = INFER_TPU_MAX_TOKENS,
    seed        = 42,
)

tokenizer = llm.get_tokenizer()
print("Model loaded.")

In [ ]:
# ── Cell 4: Load dataset + resume checkpoint ────────────────────────────
from utils import load_jsonl, load_checkpoint_indices

dataset = load_jsonl(DATASET_PATH)
done_indices = load_checkpoint_indices(CHECKPOINT_INFER)
todo = [
    {**s, "_original_position": i}
    for i, s in enumerate(dataset)
    if s["Index"] not in done_indices
]

print(f"Tổng samples   : {len(dataset)}")
print(f"Đã xử lý       : {len(done_indices)}")
print(f"Còn lại        : {len(todo)}")

In [ ]:
# ── Cell 5: Chạy inference theo batch ──────────────────────────────────
import json
from tqdm.notebook import tqdm
from utils import append_jsonl, extract_answer_from_json, extract_reasoning_from_json


def build_prompt(sample: dict) -> str:
    """Apply chat template để format prompt đúng cho từng model."""
    messages = [
        {"role": "system", "content": INFER_SYSTEM_PROMPT},
        {"role": "user",   "content": INFER_USER_TEMPLATE.format(
            question=sample["question"]
        )},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def process_batch(batch: list[dict]) -> None:
    prompts = [build_prompt(s) for s in batch]
    try:
        outputs = llm.generate(prompts, sampling_params)
    except torch.cuda.OutOfMemoryError:
        print(f"[OOM] batch size {len(batch)} — thử lại nhỏ hơn")
        if len(batch) > 1:
            mid = len(batch) // 2
            process_batch(batch[:mid])
            process_batch(batch[mid:])
        return
    except RuntimeError as e:
        print(f"[RuntimeError] {e} — skip batch indices: {[s['Index'] for s in batch]}")
        # Không ghi null — resume sẽ tự retry batch này
        return

    for sample, out in zip(batch, outputs):
        raw_output = out.outputs[0].text
        record = {
            "_original_position": sample["_original_position"],
            "index"             : sample["Index"],
            "qa_type"           : sample.get("QA_Type", ""),
            "question"          : sample["question"],
            "answer"            : sample["answer"],
            "scoring_points"    : sample.get("Scoring_Points", []),
            "raw_output"        : raw_output,
            "extracted_answer"  : extract_answer_from_json(raw_output),
            "reasoning"         : extract_reasoning_from_json(raw_output),
            "model_id"          : INFER_TPU_MODEL_ID,
            "inference_backend" : "vllm_kaggle",
        }
        append_jsonl(CHECKPOINT_INFER, record)


# ── Chạy theo batch, tqdm per batch ──
batch_size = INFER_TPU_BATCH_SIZE
batches = [todo[i:i+batch_size] for i in range(0, len(todo), batch_size)]

for batch in tqdm(batches, desc="Inference batches"):
    process_batch(batch)

print("\nInference hoàn tất.")

In [ ]:
# ── Cell 6: Kiểm tra kết quả ────────────────────────────────────────────
import json
from utils import load_jsonl

results = load_jsonl(CHECKPOINT_INFER)
done_indices = {r["index"] for r in results}
incomplete = [s for s in dataset if s["Index"] not in done_indices]

n_total  = len(results)
n_parsed = sum(1 for r in results if r.get("extracted_answer") is not None)

print(f"Hoàn thành        : {n_total} / {len(dataset)}")
print(f"Chưa xử lý        : {len(incomplete)} — indices: {[s['Index'] for s in incomplete]}")
print(f"Parse được A/B/C/D: {n_parsed} ({n_parsed/n_total*100:.1f}% trong số đã chạy)")
print()
print("Sample đầu tiên:")
print(json.dumps(results[0], ensure_ascii=False, indent=2))

## **DEBUG SECTION**

In [ ]:
import re
failed = [r for r in results if r.get("extracted_answer") is None and r.get("raw_output")]

patterns = {}
for r in failed:
    raw = r.get("raw_output", "")
    # Tìm answer field
    m = re.search(r'"answer"\s*:\s*"([^"]+)"', raw)
    val = m.group(1) if m else "NO_ANSWER_FIELD"
    patterns[val] = patterns.get(val, 0) + 1

print("Distribution of answer values in failed samples:")
for k, v in sorted(patterns.items(), key=lambda x: -x[1]):
    print(f"  '{k}': {v}")

In [ ]:
# ── Cell xuất file cuối: sort + strip _original_position ────────────────
import json as _json

OUTPUT_INFER = "/kaggle/working/inference_results.jsonl"

results = load_jsonl(CHECKPOINT_INFER)
results_sorted = sorted(results, key=lambda r: r.get("_original_position", 0))

with open(OUTPUT_INFER, "w", encoding="utf-8") as f:
    for r in results_sorted:
        clean = {k: v for k, v in r.items() if k != "_original_position"}
        f.write(_json.dumps(clean, ensure_ascii=False) + "\n")

print(f"Xuất {len(results_sorted)} records → {OUTPUT_INFER}")
print("3 records đầu (kiểm tra thứ tự):")
for r in results_sorted[:3]:
    print(f"  _pos={r['_original_position']}  index={r['index']}")